# Task 1

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
DATA_PATH = 'data'

In [27]:
def read_data(path, columns, keep, chunksize=2_000_000):
    filtered_pieces= []
    for chunk in pd.read_csv(path, sep='|', usecols=columns, chunksize=chunksize):
        filtered = chunk[keep(chunk)]
        filtered = filtered.drop_duplicates()
        filtered_pieces.append(filtered)
    result = pd.concat(filtered_pieces, ignore_index=True)
    result=result.drop_duplicates()
    return result


In [29]:
patents = read_data(f'{DATA_PATH}/202608_EPO_IPC.txt', columns=['appln_id','prio_year'],keep=lambda d: (d['prio_year']>=2000)&(d['prio_year']<=2019))

In [32]:
patents.head(5)

,appln_id,prio_year
0,8,2000
1,10,2000
2,11,2000
3,12,2000
4,13,2000


In [33]:
cpc = read_data(f'{DATA_PATH}/202608_CPC_Classes.txt', columns=['appln_id', 'CPC_Class'], keep=lambda d: d['CPC_Class'].str.startswith('Y02E', na=False) & d['appln_id'].isin(patents['appln_id']))

In [34]:
cpc.head(5)

,appln_id,CPC_Class
0,91,Y02E 30/00
1,91,Y02E 30/30
2,103,Y02E 20/16
3,146,Y02E 10/72
4,146,Y02E 10/76


In [37]:
applicants = read_data(f'{DATA_PATH}/202608_EPO_App_reg.txt', columns=['appln_id','app_name','ctry_code','app_share'],keep= lambda d: d['appln_id'].isin(patents['appln_id']))

In [38]:
applicants.head(5)

,appln_id,app_name,ctry_code,app_share
0,8,BABCOCK-HITACHI KABUSHIKI KAISHA,JP,1.0
1,10,Saltigo GmbH,DE,1.0
2,11,HONDA GIKEN KOGYO KABUSHIKI KAISHA,JP,1.0
3,12,Institut für Angewandte Chemie Berlin-Adlersho...,DE,1.0
4,13,NEC Electronics Corporation,JP,1.0


### check the data

In [42]:
print(f'The number of patent applications: {len(patents):,.0f}')
print(f'The number of green patent applications: {cpc['appln_id'].nunique():,.0f}')
print(f'The sum of all app_share is: {applicants['app_share'].sum():,.0f}')

The number of patent applications: 2,835,599
The number of green patent applications: 114,924
The sum of all app_share is: 2,835,140


This checks out with correct numbers from the assignment description

In [47]:
print(f'The average number of Y02E codes per green patent: {cpc.groupby('appln_id').size().mean():.2f}')

cpc['main_group'] = cpc['CPC_Class'].str.partition('/')[0].str[-2:]

groups_per_patent = cpc.groupby('appln_id')['main_group'].nunique()

print(f'Green patents in more than one main group: {(groups_per_patent > 1).sum():,}')

countries_per_patent = applicants.groupby('appln_id')['ctry_code'].nunique()

print(f'Number of patents with applicants in more than one country: {(countries_per_patent > 1).sum():,}')

The average number of Y02E codes per green patent: 1.23
Green patents in more than one main group: 4,359
Number of patents with applicants in more than one country: 58,954
